In [ ]:
# pip install fairlearn
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate
from sklearn.metrics import accuracy_score
import pandas as pd

# Load predictions
df_test = X_test.copy()
df_test['actual']    = y_test.values
df_test['predicted'] = (model.predict_proba(X_test)[:, 1] > 0.3).astype(int)

# Create age band for fairness testing
df_test['age_band'] = pd.cut(
    df_test['AGE_YEARS'],
    bins=[18, 30, 45, 60, 100],
    labels=['18-30', '31-45', '46-60', '60+']
)

# Compute fairness metrics across age groups
metric_frame = MetricFrame(
    metrics={
        'accuracy':      accuracy_score,
        'selection_rate': selection_rate,
        'fpr':           false_positive_rate,
    },
    y_true=df_test['actual'],
    y_pred=df_test['predicted'],
    sensitive_features=df_test['age_band']
)

print('=== Fairness Metrics by Age Group ===')
print(metric_frame.by_group)
print()
print(f'Max Disparity (Accuracy): {metric_frame.difference()["accuracy"]:.3f}')
# Target: disparity < 0.05 for regulatory compliance
